In [125]:
:dep smartcore = { version = "0.3.2", features = ["ndarray-bindings", "serde"] }
:dep bincode = "1.3"
:dep csv = "1.1"
:dep ndarray = { version = "0.15", features = ["rayon"] }
:dep linfa = { version = "0.7" }
:dep rand = "0.8"
:dep rand_chacha = "0.3"
:dep plotters = "0.3"
:dep linfa-preprocessing = "0.7"
:dep linfa-elasticnet = { version = "0.7"}
:opt 3

Optimization: 3


In [176]:
use csv;
use linfa::{Dataset};
use linfa::prelude::*;
use ndarray::{Array1, Array2, ArrayView1, ArrayView2, Axis, Ix1, array,s};
use rand::SeedableRng;
use rand_chacha::ChaCha12Rng;
use linfa_preprocessing::linear_scaling::LinearScaler;
use linfa::traits::{Fit, Transformer};
use linfa_elasticnet::{ElasticNet};
use ndarray::arr0;
use linfa_preprocessing::linear_scaling::LinearScaler;
use ndarray::{Array2, concatenate, Axis};

In [137]:
let rec_path= "train_n.csv";
let targ_path= "train_n_y.csv";
let mut reader_rec =csv::Reader::from_path(rec_path).unwrap();
let mut reader_targ =csv::Reader::from_path(targ_path).unwrap();
let mut records: Array2<f64> = Array2::zeros((0, 22));  //X
let mut targets: Array1<f64>= Array1::zeros(0); //Y
let mut rng = ChaCha12Rng::seed_from_u64(5);

In [138]:
for data in reader_rec.records(){
        let rec = data.unwrap();
        let row_vec: Vec<f64> = (0..22)
        .map(|i| rec[i].parse::<f64>().expect("pupupu"))
        .collect();
        let array = Array2::from_shape_vec((1, 22), row_vec).expect("pupep");
        let _  = records.append(Axis(0), array.view());
}
for data in reader_targ.records(){
        let targ = data.unwrap();
        let _ = targets.append(Axis(0),array![targ[0].parse::<f64>().expect("pupupu")].view());
}
let mut data_set= Dataset::new(records.clone(), targets.clone());
let (train_data,test_data) = data_set.shuffle(&mut rng)
                                     .split_with_ratio(0.85);

In [151]:
fn expand_to_poly(records: &Array2<f64>, degree: i32) -> Array2<f64> {
    let mut expanded = records.clone();
    for d in 2..=degree {
        for column in records.columns() {
            let poly_col = column.mapv(|x| x.powi(d)).insert_axis(Axis(1));
            expanded = concatenate![Axis(1), expanded, poly_col];
        }
    }
    expanded
}
fn create_and_train_model(
    ds: &Dataset<f64, f64, Ix1>, 
    degree: i32, 
    penalty: f64, 
    l1_ratio: f64
) -> (ElasticNet<f64>, LinearScaler<f64>) {
    let poly_records = expand_to_poly(ds.records(), degree);
    let targets = ds.targets().to_owned();
    let poly_ds = Dataset::new(poly_records, targets);
    let scaler = LinearScaler::standard().fit(&poly_ds).unwrap();
    let scaled_ds = scaler.transform(poly_ds);
    let model = ElasticNet::params()
        .penalty(penalty)
        .l1_ratio(l1_ratio)
        .fit(&scaled_ds)
        .unwrap();

    (model, scaler)
}

In [146]:
let degrees = vec![1, 2, 3]; 
let penalties = vec![0.0001, 0.00005, 0.00001, 0.000025];
let mut best_r2 = -f64::INFINITY;
let mut best_params = (0, 0.0);

for &deg in &degrees {
    let poly_records = expand_to_poly(data_set.records(), deg);
    let targets = data_set.targets().to_owned();
    let poly_ds = Dataset::new(poly_records, targets);

    let scaler = LinearScaler::standard().fit(&poly_ds).expect("pupeu");
    let scaled_ds_full = scaler.transform(poly_ds);

    // 1. Проверяем количество строк
    let n_rows = scaled_ds_full.records().nrows();
    let n_folds = 5;
    let final_rows = n_rows - (n_rows % n_folds);
    let mut scaled_ds = Dataset::new(
        scaled_ds_full.records().slice(s![0..final_rows, ..]).to_owned(),
        scaled_ds_full.targets().slice(s![0..final_rows]).to_owned()
    );

    let models: Vec<_> = penalties.iter()
        .map(|&p| ElasticNet::params().penalty(p).l1_ratio(1.0).max_iterations(100))
        .collect();
    let r2_scores = scaled_ds.cross_validate(n_folds, &models, |prediction, truth| {
        prediction.r2(truth).map(arr0)
    }).expect("pepsihs");
    let mean_r2_per_model = r2_scores.mean_axis(ndarray::Axis(0)).unwrap();
    for (i, &m_r2) in mean_r2_per_model.iter().enumerate() {
        if m_r2 > best_r2 {
            best_r2 = m_r2;
            best_params = (deg, penalties[i]);
        }
    }
    println!("{:?},{:?}", best_r2,best_params);
}


0.5504743992700324,(1, 0.0001)
0.6802623803627152,(2, 0.0001)
0.6802623803627152,(2, 0.0001)


()

In [152]:
let final_degree = 2;
let final_penalty = 0.0001;
let (final_model,final_scaler) =  create_and_train_model(&train_data,final_degree,final_penalty,1.0);

In [154]:
let n_features = final_model.hyperplane().len();
let significant_features = final_model.hyperplane()
    .iter()
    .filter(|&&w| w.abs() > 1.0)
    .count();
println!("Кол-во признаков после масштабирования: {}", n_features);
println!("Кол-во значимых признаков: {}", significant_features);


Кол-во признаков после масштабирования: 44
Кол-во значимых признаков: 0


In [177]:
let weights_vec = final_model.hyperplane().to_owned();
let max_weight = weights_vec.iter().map(|w| w.abs()).fold(0.0, f64::max);
let non_zero = weights_vec.iter().filter(|&&w| w.abs() > 1e-5).count();
println!("Максимальный абсолютный вес: {:.5}", max_weight);
println!("Ненулевых признаков: {} из {}", non_zero, weights_vec.len());

Максимальный абсолютный вес: 0.61294
Ненулевых признаков: 43 из 44


In [178]:
let poly_test_records = expand_to_poly(test_data.records(), 2);
let poly_test_ds = Dataset::new(poly_test_records, test_data.targets().to_owned());
let scaled_test_ds = final_scaler.transform(poly_test_ds);
let predictions = final_model.predict(&scaled_test_ds);
let test_r2 = predictions.r2(&scaled_test_ds).unwrap();
let y_pred = final_model.predict(&scaled_test_ds).to_owned();
let y_true = scaled_test_ds.targets().to_owned();
let mse = y_true.iter()
    .zip(y_pred.iter())
    .map(|(yt, yp)| (yt - yp).powi(2))
    .sum::<f64>() / y_true.len() as f64;
let test_rmse = mse.sqrt();

println!("RMSE: {:.5}", test_rmse);
println!("R2: {:.4}", test_r2);


RMSE: 0.27992
R2: 0.6811


In [175]:
let y_pred = final_model.predict(&scaled_test_ds).to_owned();
let y_true = scaled_test_ds.targets().to_owned();
let residuals: Vec<f64> = y_true.iter()
    .zip(y_pred.iter())
    .map(|(yt, yp)| yt - yp)
    .collect();
let mean_res = residuals.iter().sum::<f64>() / residuals.len() as f64;
let max_res = residuals.iter().fold(f64::NEG_INFINITY, |a, &b| a.max(b));
let min_res = residuals.iter().fold(f64::INFINITY, |a, &b| a.min(b));
let threshold = 2.0 * 0.27992;
let outliers = residuals.iter().filter(|r| r.abs() > threshold).count();
println!("Среднее значение остатков: {:.6}", mean_res);
println!("Максимальная ошибка: {:.4}", max_res);
println!("Минимальная ошибка:  {:.4}", min_res);
println!("Количество выбросов: {} ({:.3}%)", outliers, (outliers as f64 / residuals.len() as f64) * 100.0);

Среднее значение остатков: 0.002075
Максимальная ошибка: 1.1665
Минимальная ошибка:  -1.3269
Количество выбросов: 828 (5.313%)
